In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Environment Configuration (Keep these)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop" 
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.path.pathsep + os.environ["PATH"]

def create_spark_session():
    WAREHOUSE_DIR = os.path.abspath("output/iceberg_warehouse")
    return SparkSession.builder \
        .appName("RetailLakehouse") \
        .master("local[*]") \
        .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.local.type", "hadoop") \
        .config("spark.sql.catalog.local.warehouse", WAREHOUSE_DIR) \
        .config("spark.sql.defaultCatalog", "local") \
        .getOrCreate()

spark = create_spark_session()

print("======================================================")
print("SUCCESS: Spark Session connected successfully!")
print(f"Spark Version: {spark.version}")
print(f"Warehouse Location: {os.path.abspath('output/iceberg_warehouse')}")
print("======================================================")


SUCCESS: Spark Session connected successfully!
Spark Version: 3.5.4
Warehouse Location: c:\Users\palisha.shakya\Downloads\Project1\Iceberg\output\iceberg_warehouse


In [2]:
from pyspark.sql.functions import col

# Load
df = spark.read.option("header", "true").csv("../Data/fact_sales.csv")

# Transform
df_transformed = df.select(
    col("order_id").cast("long"),
    col("order_date").cast("date"),
    col("customer_id").cast("long"),
    col("product_id").cast("long"),
    col("store_id").cast("long"),
    col("quantity").cast("long"),
    col("total_amount").cast("double")
)

# Write to Iceberg
df_transformed.write.mode("overwrite").saveAsTable("local.db.fact_sales")
print("Data loaded to Iceberg successfully.")

Data loaded to Iceberg successfully.


In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.db")
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.db.fact_sales (
        order_id bigint, 
        order_date date, 
        customer_id bigint, 
        product_id bigint, 
        store_id bigint, 
        quantity bigint, 
        total_amount double
    ) 
    USING iceberg 
    PARTITIONED BY (days(order_date))
""")

DataFrame[]

In [26]:
from pyspark.sql.functions import col

# ------------------------------------------------------------------
# 1. Read CSV
# ------------------------------------------------------------------
df = spark.read.option("header", "true").csv("../Data/fact_sales.csv")

# ------------------------------------------------------------------
# 2. Transform data types
# ------------------------------------------------------------------
df_transformed = df.select(
    col("order_id").cast("bigint").alias("order_id"),
    col("order_date").cast("date").alias("order_date"),
    col("customer_id").cast("bigint").alias("customer_id"),
    col("product_id").cast("bigint").alias("product_id"),
    col("store_id").cast("bigint").alias("store_id"),
    col("quantity").cast("bigint").alias("quantity"),
    col("total_amount").cast("double").alias("total_amount")
)

# ------------------------------------------------------------------
# 3. Remove existing unpartitioned table
# ------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS local.db.fact_sales")

# ------------------------------------------------------------------
# 4. Create a NEW partitioned Iceberg table
# ------------------------------------------------------------------
spark.sql("""
CREATE TABLE local.db.fact_sales (
    order_id BIGINT,
    order_date DATE,
    customer_id BIGINT,
    product_id BIGINT,
    store_id BIGINT,
    quantity BIGINT,
    total_amount DOUBLE
)
USING iceberg
PARTITIONED BY (days(order_date))
""")

# ------------------------------------------------------------------
# 5. Load data into the table
# ------------------------------------------------------------------
df_transformed.writeTo("local.db.fact_sales").append()

print("Partitioned Iceberg table created successfully!")

Partitioned Iceberg table created successfully!


In [27]:
spark.sql("DESCRIBE EXTENDED local.db.fact_sales").show(200, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|order_id                    |bigint                                                                                                                |NULL   |
|order_date                  |date                                                                                                                  |NULL   |
|customer_id                 |bigint                                                                                                                |NULL   |
|product_id                  |bigint                

In [28]:
spark.sql("""
EXPLAIN
SELECT *
FROM local.db.fact_sales
WHERE order_date = DATE '2026-06-29'
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [29]:
spark.sql("""
SELECT file_path, partition
FROM local.db.fact_sales.files
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|file_path                                                                                                                                                                             |partition   |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-09-16/00000-12-7247d207-fb07-4eef-805c-2aed77711a94-0-00052.parquet|{2026-09-16}|
|c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-09-17/00000-12-7247d207-fb07-4eef-805c-2aed77711a94-0-00119.parquet|{2026-09-17}|
|c:/Users/

In [30]:
spark.sql("""
SELECT order_date, COUNT(*)
FROM local.db.fact_sales
GROUP BY order_date
ORDER BY order_date
""").show()

+----------+--------+
|order_date|count(1)|
+----------+--------+
|2026-01-02|       2|
|2026-01-04|       3|
|2026-01-05|       3|
|2026-01-06|       2|
|2026-01-07|       1|
|2026-01-08|       2|
|2026-01-09|       3|
|2026-01-10|       2|
|2026-01-11|       3|
|2026-01-12|       2|
|2026-01-13|       1|
|2026-01-14|       2|
|2026-01-15|       5|
|2026-01-16|       4|
|2026-01-17|       4|
|2026-01-18|       2|
|2026-01-19|       3|
|2026-01-20|       1|
|2026-01-21|       2|
|2026-01-22|       1|
+----------+--------+
only showing top 20 rows



In [31]:
spark.sql("""
SELECT COUNT(DISTINCT order_date)
FROM local.db.fact_sales
""").show()

+--------------------------+
|count(DISTINCT order_date)|
+--------------------------+
|                       335|
+--------------------------+



In [32]:
spark.sql("""
SELECT
partition,
record_count,
file_path
FROM local.db.fact_sales.files
ORDER BY partition
""").show(100, truncate=False)

+------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|partition   |record_count|file_path                                                                                                                                                                             |
+------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{2026-01-02}|2           |c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-01-02/00000-12-7247d207-fb07-4eef-805c-2aed77711a94-0-00040.parquet|
|{2026-01-04}|3           |c:/Users/palisha.shakya/Downloads/Project1/Iceberg/output/iceberg_warehouse/db/fact_sales/data/order_date_day=2026-01-04/00000-12

In [33]:
spark.sql("""
SELECT COUNT(*)
FROM local.db.fact_sales.files
""").show()

+--------+
|count(1)|
+--------+
|     335|
+--------+



In [36]:
spark.sql("""
EXPLAIN
SELECT *
FROM local.db.fact_sales
WHERE order_date = DATE '2026-06-29'
""").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                                          |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [37]:
spark.sql("""
EXPLAIN FORMATTED
SELECT *
FROM local.db.fact_sales
WHERE order_date = DATE '2026-06-29'
""").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                           